# 2.3 類別資料編碼

這份 Notebook 示範 pandas one-hot encoding，以及 Keras `StringLookup` 類別前處理。

## 1. 載入套件與建立資料

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

tf.keras.utils.set_random_seed(42)
rng = np.random.default_rng(42)

n = 900
city = rng.choice(['taipei', 'taichung', 'tainan', 'kaohsiung'], size=n, p=[0.35, 0.25, 0.2, 0.2])
device = rng.choice(['mobile', 'desktop', 'tablet'], size=n, p=[0.5, 0.35, 0.15])
age = rng.normal(35, 9, n).clip(18, 70)
visits = rng.poisson(4, n)
logit = (city == 'taipei') * 0.8 + (device == 'desktop') * 0.5 + 0.05 * (age - 35) + 0.15 * (visits - 4)
y = (logit + rng.normal(0, 0.8, n) > 0.5).astype('int64')

df = pd.DataFrame({'city': city, 'device': device, 'age': age, 'visits': visits, 'target': y})
df.head()

## 2. pandas one-hot encoding

In [ ]:
pd.get_dummies(df[['city', 'device']], dtype='int64').head()

## 3. 切分資料

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.25, random_state=42, stratify=df['target'])

numeric_train = train_df[['age', 'visits']].astype('float32').values
numeric_test = test_df[['age', 'visits']].astype('float32').values

y_train = train_df['target'].astype('float32').values
y_test = test_df['target'].astype('float32').values

## 4. 建立 Keras 類別前處理 layer

In [ ]:
city_lookup = tf.keras.layers.StringLookup(output_mode='one_hot')
device_lookup = tf.keras.layers.StringLookup(output_mode='one_hot')
city_lookup.adapt(train_df['city'].values)
device_lookup.adapt(train_df['device'].values)

print('city vocabulary:', city_lookup.get_vocabulary())
print('device vocabulary:', device_lookup.get_vocabulary())

## 5. 建立可處理數值與類別欄位的模型

In [ ]:
city_input = tf.keras.Input(shape=(1,), dtype=tf.string, name='city')
device_input = tf.keras.Input(shape=(1,), dtype=tf.string, name='device')
numeric_input = tf.keras.Input(shape=(2,), dtype=tf.float32, name='numeric')

city_encoded = city_lookup(city_input)
device_encoded = device_lookup(device_input)
features = tf.keras.layers.Concatenate()([numeric_input, city_encoded, device_encoded])
x = tf.keras.layers.Dense(16, activation='relu')(features)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)
model = tf.keras.Model(inputs={'city': city_input, 'device': device_input, 'numeric': numeric_input}, outputs=output)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## 6. 訓練與評估

In [ ]:
train_inputs = {
    'city': tf.constant(train_df['city'].astype(str).values.reshape(-1, 1)),
    'device': tf.constant(train_df['device'].astype(str).values.reshape(-1, 1)),
    'numeric': numeric_train,
}
test_inputs = {
    'city': tf.constant(test_df['city'].astype(str).values.reshape(-1, 1)),
    'device': tf.constant(test_df['device'].astype(str).values.reshape(-1, 1)),
    'numeric': numeric_test,
}

model.fit(train_inputs, y_train, validation_split=0.2, epochs=25, batch_size=32, verbose=0)

pd.DataFrame([
    model.evaluate(train_inputs, y_train, verbose=0, return_dict=True),
    model.evaluate(test_inputs, y_test, verbose=0, return_dict=True),
], index=['train', 'test'])

## 7. 未知類別測試

In [ ]:
new_data = {
    'city': tf.constant(np.array(['taipei', 'hsinchu']).reshape(-1, 1)),
    'device': tf.constant(np.array(['mobile', 'smart_tv']).reshape(-1, 1)),
    'numeric': np.array([[30, 3], [45, 8]], dtype='float32')
}
model.predict(new_data, verbose=0)

## 8. 小結

Keras lookup layer 可以處理未知類別，並讓前處理跟模型一起保存。